# Tutorial 02: Real-World Maps with OSMnx

This tutorial demonstrates how to create VRP problem instances using **real-world street networks** from OpenStreetMap via OSMnx.

**What you'll learn:**
- Load street networks from OpenStreetMap
- Map locations to network nodes
- Compute network-based distance matrices (instead of Euclidean)
- Create PDPTW instances with real geographic data
- Visualize routes on actual streets

**Example location:** Purdue University campus

## 1. Setup and Imports

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# VRP Toolkit imports
from vrp_toolkit.data import (
    OSMnxNetworkLoader,
    map_locations_to_nodes,
    compute_distance_matrix,
    compute_time_matrix,
    create_pdptw_order_table_from_locations,
    visualize_routes_on_network,
    create_pdptw_from_osm  # Convenience function
)
from vrp_toolkit.problems.pdptw import PDPTWInstance
from vrp_toolkit.algorithms.alns.solver import ALNS, ALNSConfig, greedy_insertion_initial_solution

# Check OSMnx availability
try:
    import osmnx as ox
    print(f"OSMnx version: {ox.__version__}")
except ImportError:
    print("ERROR: OSMnx not installed. Install with: pip install osmnx")
    raise

## 2. Quick Start: Using the Convenience Function

The easiest way to create a real-world PDPTW instance is using `create_pdptw_from_osm()`:

In [ ]:
# Define locations (latitude, longitude)
place_name = "Purdue University, West Lafayette, IN, USA"

depot_location = (40.4237, -86.9212)  # Purdue Memorial Union

# Example pickup and delivery locations around campus
pickup_locations = [
    (40.4280, -86.9145),  # Near Engineering buildings
    (40.4200, -86.9180),  # Near residence halls
]

delivery_locations = [
    (40.4250, -86.9100),  # Near Student Union
    (40.4210, -86.9220),  # Near Athletic facilities
]

charging_location = (40.4237, -86.9212)  # Same as depot

print(f"Creating PDPTW instance for: {place_name}")
print(f"Depot: {depot_location}")
print(f"Pickups: {len(pickup_locations)}")
print(f"Deliveries: {len(delivery_locations)}")

In [ ]:
# Create complete PDPTW instance from OSM
# This will download the street network (and cache it for reuse)
order_table, distance_matrix, time_matrix, G, node_mapping = create_pdptw_from_osm(
    place_name=place_name,
    depot_location=depot_location,
    pickup_locations=pickup_locations,
    delivery_locations=delivery_locations,
    charging_location=charging_location,
    cache_file="../data/purdue_network.graphml",  # Cache for faster reuse
    demand=10.0,
    time_window=(0.0, 480.0),  # 8 hours
    service_time=5.0  # 5 minutes
)

print("\n" + "="*50)
print("PDPTW Instance Created Successfully!")
print("="*50)

## 3. Inspect the Created Instance

In [ ]:
# View order table
print("Order Table:")
print(order_table)

print(f"\nDistance Matrix Shape: {distance_matrix.shape}")
print(f"Distance Matrix (meters):\n{distance_matrix.astype(int)}")

print(f"\nTime Matrix Shape: {time_matrix.shape}")
print(f"Time Matrix (minutes):\n{time_matrix.round(2)}")

## 4. Create PDPTWInstance and Solve

In [ ]:
# Create PDPTW instance
instance = PDPTWInstance(
    order_table=order_table,
    distance_matrix=distance_matrix,
    time_matrix=time_matrix,
    robot_speed=1.0  # Speed is already factored into time_matrix
)

print(f"PDPTWInstance created:")
print(f"  - Nodes: {instance.num_nodes}")
print(f"  - Orders: {instance.num_orders}")
print(f"  - Depot: {instance.depot_index}")

In [ ]:
# Create initial solution
initial_solution = greedy_insertion_initial_solution(
    problem=instance,
    num_vehicles=2,
    vehicle_capacity=50.0,
    battery_capacity=100.0,
    battery_consume_rate=0.1,
    penalty_unvisit=1000.0,
    penalty_delay=100.0
)

print(f"Initial solution created:")
print(f"  - Objective: {initial_solution.objective_function():.2f}")
print(f"  - Feasible: {initial_solution.is_feasible()}")
print(f"  - Routes: {initial_solution.routes}")

In [ ]:
# Solve with ALNS
print("Solving with ALNS...")

config = ALNSConfig(
    max_no_improve=50,
    segment_length=10
)

solver = ALNS(
    initial_solution=initial_solution,
    dist_matrix=instance.distance_matrix,
    battery_capacity=100.0,
    config=config
)

solution = solver.solve()

print(f"\nSolution found:")
print(f"  - Objective: {solution.objective_function():.2f}")
print(f"  - Feasible: {solution.is_feasible()}")
print(f"  - Routes: {solution.routes}")

## 5. Visualize Routes on Real Street Network

The real power of OSMnx integration: visualizing routes on actual streets!

In [ ]:
# Visualize solution routes overlaid on street network
visualize_routes_on_network(
    G=G,
    routes=solution.routes,
    node_mapping=node_mapping,
    title="VRP Solution on Purdue Campus Street Network",
    save_path="../figures/purdue_vrp_solution.png"
)

## 6. Step-by-Step Workflow (Advanced)

For more control, you can execute each step manually:

### Step 1: Load Street Network

In [ ]:
# Manual approach: Load network step by step
loader = OSMnxNetworkLoader()

# Load by place name (with caching)
G_manual = loader.load_by_place(
    place_name="Purdue University, West Lafayette, IN, USA",
    network_type='drive',
    cache_file="../data/purdue_network_manual.graphml"
)

# Ensure connectivity
G_manual = loader.ensure_connectivity()

print(f"Network loaded: {len(G_manual.nodes)} nodes, {len(G_manual.edges)} edges")

### Step 2: Map Locations to Network Nodes

In [ ]:
# Define all locations
all_locations = [depot_location] + pickup_locations + delivery_locations + [charging_location]

# Map to nearest network nodes
all_nodes = map_locations_to_nodes(G_manual, all_locations)

depot_node = all_nodes[0]
pickup_nodes = all_nodes[1:3]
delivery_nodes = all_nodes[3:5]
charging_node = all_nodes[5]

print(f"Depot node (OSM ID): {depot_node}")
print(f"Pickup nodes: {pickup_nodes}")
print(f"Delivery nodes: {delivery_nodes}")
print(f"Charging node: {charging_node}")

### Step 3: Compute Distance and Time Matrices

In [ ]:
# Compute network-based distance matrix
distance_matrix_manual = compute_distance_matrix(G_manual, all_nodes, weight='length')

# Convert to time matrix (assuming 30 km/h average speed)
time_matrix_manual = compute_time_matrix(distance_matrix_manual, average_speed_kmh=30)

print(f"Distance matrix computed: {distance_matrix_manual.shape}")
print(f"Sample distances (meters): {distance_matrix_manual[0, :].astype(int)}")
print(f"\nTime matrix computed: {time_matrix_manual.shape}")
print(f"Sample times (minutes): {time_matrix_manual[0, :].round(2)}")

### Step 4: Create Order Table

In [ ]:
# Create order table from OSM nodes
order_table_manual = create_pdptw_order_table_from_locations(
    depot_node=depot_node,
    pickup_nodes=pickup_nodes,
    delivery_nodes=delivery_nodes,
    G=G_manual,
    charging_node=charging_node,
    demand=10.0,
    time_window=(0.0, 480.0),
    service_time=5.0
)

print("Order table created:")
print(order_table_manual)

## 7. Comparison: Network Distance vs Euclidean Distance

Let's compare network-based distances with straight-line (Euclidean) distances:

In [ ]:
# Compute Euclidean distances for comparison
coords = np.array([[row['X'], row['Y']] for _, row in order_table.iterrows()])
euclidean_matrix = np.sqrt(((coords[:, None, :] - coords[None, :, :])**2).sum(axis=2))

# Compare
print("Distance Comparison (meters):")
print("\nNetwork-based distances:")
print(distance_matrix[:3, :3].astype(int))
print("\nEuclidean distances (scaled to meters):")
print((euclidean_matrix[:3, :3] * 100000).astype(int))  # Scale lat/lon to approx meters

# Calculate ratio
ratio = distance_matrix / (euclidean_matrix * 100000 + 1)  # +1 to avoid division by zero
print(f"\nAverage network/Euclidean ratio: {ratio[ratio < 10].mean():.2f}")
print("(Network distances are typically 1.2-1.5x longer than straight-line)")

## 8. Key Takeaways

**What we learned:**
1. ✅ OSMnx integration makes VRP problems more realistic
2. ✅ Network-based distances account for actual road layouts
3. ✅ Caching networks improves performance
4. ✅ Visualizing routes on real streets provides better insights

**When to use real maps:**
- Research papers with real-world validation
- Practical applications (delivery, logistics)
- Demonstrations and presentations

**When to use synthetic maps:**
- Benchmarking algorithms
- Controlled experiments
- Quick prototyping

**Next steps:**
- Try different locations (cities, neighborhoods)
- Experiment with different network types (walk, bike)
- Compare solutions on synthetic vs real maps
- Use larger instances with more pickups/deliveries

## 9. Exercise: Try Your Own Location

Modify the cells above to create a VRP instance for your own location:
1. Change `place_name` to your city/campus/neighborhood
2. Define custom pickup and delivery locations
3. Solve and visualize the routes

**Tips:**
- Use Google Maps to find lat/lon coordinates
- Start with small areas (faster downloads)
- Cache your network for reuse
- Experiment with different speeds and time windows